In [1]:
from datasets import load_dataset

ds = load_dataset("code-search-net/code_search_net", "python")
train_set = ds["train"]
val_set = ds["validation"]
test_set = ds["test"]
display(train_set)

/Users/kuzeyaldemir/Projects/tiny-coder/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
    num_rows: 412178
})

In [2]:
corpus = "\n\n".join(test_set['whole_func_string'])
print(f"Length of the corpus in chars: {len(corpus):,}")

Length of the corpus in chars: 23,711,286


In [ ]:
from bpe_tokenizer import BPETokenizer
from time import perf_counter

tokenizer = BPETokenizer()

start_time = perf_counter()

tokenizer.train(
    training_text=corpus,
    vocab_size=300,
    min_pair_frequency=2
)

elapsed_time = perf_counter() - start_time
print(f"Training took {elapsed_time:.2f} seconds")

Training took 170.79 seconds


Training the tokenizer with test set corpus with 260 vocab size and 2 min frequency took 21.5 seconds.
Training the tokenizer with test set corpus with 300 vocab size and 2 min frequency took 170.7 seconds.

In [7]:
from tokenizers import ByteLevelBPETokenizer
from tokenizers.pre_tokenizers import ByteLevel
from time import perf_counter

def batch_iterator(
        dataset,
        column: str = "whole_func_string",
        batch_size: int = 1000
):
    text_dataset = dataset.select_columns(column)
    for batch in text_dataset.iter(batch_size=batch_size):
        texts = [
            text
            for text in batch[column]
        ]
        yield texts


iterator = batch_iterator(test_set)

hf_tokenizer = ByteLevelBPETokenizer(
    add_prefix_space=False,
)
hf_tokenizer.pre_tokenizer = ByteLevel(
    add_prefix_space=False,
    use_regex=False,
)

start_time = perf_counter()
hf_tokenizer.train_from_iterator(
    iterator,
    vocab_size=300,
    min_frequency=10,
    show_progress=True,
    length=(len(test_set))
)

elapsed_time = perf_counter() - start_time
print(f"Training time: {elapsed_time:.2f} seconds")
print(f"Actual vocabulary size: {hf_tokenizer.get_vocab_size()}")




Training time: 7.80 seconds
Actual vocabulary size: 300


Training the tokenizer with the hugging face byteleveltokenizer produced 7 second training time compared to 170 seconds on our own implementation